# Genesis Mission Funding and Collaboration Dashboard

This dashboard summarizes BASELINE, UFR, and ROT funding by Genesis Mission Challenge, program office contribution, and project funding disruptiveness. Validation, AI-directness, and confidence details are retained as traceability appendix material in the exported static HTML report.


In [ ]:
from pathlib import Path
from html import escape
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from programming_ontology import paths
from programming_ontology.asksage_classifier import (
    default_credentials_path,
    resolve_asksage_openai_base_url,
)
from programming_ontology.classify_programming import run_classification_pipeline

paths.ensure_output_dirs()

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 220)
plt.style.use("default")


def format_money(value):
    value = 0 if pd.isna(value) else float(value)
    sign = "-" if value < 0 else ""
    value = abs(value)
    if value >= 1_000_000_000:
        return f"{sign}${value / 1_000_000_000:,.2f}B"
    if value >= 1_000_000:
        return f"{sign}${value / 1_000_000:,.1f}M"
    return f"{sign}${value:,.0f}"


def bool_env(name, default=False):
    raw = os.environ.get(name, "").strip().casefold()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


def optional_int_env(name):
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else None


In [ ]:
REQUIRED_ARTIFACT_PATHS = {
    "row-level classifications": paths.PROGRAMMING_CHALLENGE_CLASSIFICATIONS_PATH,
    "reviewed classifications": paths.REVIEWED_PROGRAMMING_CHALLENGE_CLASSIFICATIONS_PATH,
    "office claim validation": paths.OFFICE_CLAIM_VALIDATION_PATH,
    "challenge coverage summary": paths.CHALLENGE_COVERAGE_SUMMARY_PATH,
    "office challenge evidence": paths.OFFICE_CHALLENGE_EVIDENCE_PATH,
    "WBS challenge summary": paths.WBS_CHALLENGE_SUMMARY_PATH,
    "classified ontology graph": paths.CLASSIFIED_ONTOLOGY_GRAPH_PATH,
}


def artifact_ready(path):
    return path.exists() and path.stat().st_size > 0


missing_artifacts = [
    name for name, path in REQUIRED_ARTIFACT_PATHS.items() if not artifact_ready(path)
]
REFRESH_ARTIFACTS_WITH_ASKSAGE = bool_env("REFRESH_ARTIFACTS_WITH_ASKSAGE", default=False)
AUTO_REFRESH_MISSING_ARTIFACTS = bool_env("AUTO_REFRESH_MISSING_ARTIFACTS", default=True)
ASKSAGE_TOP_K = int(os.environ.get("ASKSAGE_TOP_K", "5"))
ASKSAGE_LIMIT = optional_int_env("ASKSAGE_LIMIT")
ASKSAGE_REASONING_EFFORT = os.environ.get("ASKSAGE_REASONING_EFFORT", "medium").strip()
ASKSAGE_CREDENTIALS_PATH = default_credentials_path()
ASKSAGE_MODEL = os.environ.get("ASKSAGE_MODEL", "").strip() or None
ASKSAGE_OPENAI_BASE_URL = resolve_asksage_openai_base_url(credentials_path=ASKSAGE_CREDENTIALS_PATH)

should_refresh = REFRESH_ARTIFACTS_WITH_ASKSAGE or (
    AUTO_REFRESH_MISSING_ARTIFACTS and bool(missing_artifacts)
)

print(f"Using Ask Sage credentials file: {ASKSAGE_CREDENTIALS_PATH}")
print(f"Resolved OpenAI SDK base URL: {ASKSAGE_OPENAI_BASE_URL}")

if should_refresh:
    if missing_artifacts:
        display(Markdown("Refreshing because required artifacts are missing: " + ", ".join(missing_artifacts)))
    else:
        display(Markdown("Refreshing artifacts with Ask Sage because `REFRESH_ARTIFACTS_WITH_ASKSAGE` is enabled."))
    refresh_outputs = run_classification_pipeline(
        classifier="asksage",
        model=ASKSAGE_MODEL,
        top_k=ASKSAGE_TOP_K,
        reasoning_effort=ASKSAGE_REASONING_EFFORT,
        limit=ASKSAGE_LIMIT,
        resume_cache=True,
        audit_cache_path=paths.ASKSAGE_CLASSIFICATION_AUDIT_PATH,
        ai_directness_audit_cache_path=paths.ASKSAGE_AI_DIRECTNESS_AUDIT_PATH,
        write_outputs=True,
    )
    display(Markdown(f"Refreshed **{len(refresh_outputs['classifications']):,}** rows with Ask Sage."))
else:
    if missing_artifacts:
        raise FileNotFoundError(
            "Required dashboard artifacts are missing and auto-refresh is disabled: "
            + ", ".join(missing_artifacts)
        )
    display(Markdown("Using existing processed artifacts. Set `REFRESH_ARTIFACTS_WITH_ASKSAGE=1` to regenerate them with Ask Sage."))


In [ ]:
classifications = pd.read_csv(paths.PROGRAMMING_CHALLENGE_CLASSIFICATIONS_PATH)
reviewed_classifications = pd.read_csv(paths.REVIEWED_PROGRAMMING_CHALLENGE_CLASSIFICATIONS_PATH)
office_claim_validation = pd.read_csv(paths.OFFICE_CLAIM_VALIDATION_PATH)
challenge_coverage = pd.read_csv(paths.CHALLENGE_COVERAGE_SUMMARY_PATH)
office_challenge_evidence = pd.read_csv(paths.OFFICE_CHALLENGE_EVIDENCE_PATH)
wbs_challenge_summary = pd.read_csv(paths.WBS_CHALLENGE_SUMMARY_PATH)

for frame in [classifications, reviewed_classifications, office_claim_validation, challenge_coverage, office_challenge_evidence, wbs_challenge_summary]:
    if "five_year_total" in frame.columns:
        frame["five_year_total"] = pd.to_numeric(frame["five_year_total"], errors="coerce").fillna(0.0)

ai_scored_rows = reviewed_classifications.get("ai_directness_score", pd.Series(dtype=float))
ai_scored_rows = pd.to_numeric(ai_scored_rows, errors="coerce").dropna()
average_ai_score = "n/a" if ai_scored_rows.empty else f"{ai_scored_rows.mean():.1f}"

display(Markdown(f"""
## Loaded Artifacts

- Raw classifier rows (diagnostic): **{len(classifications):,}**
- Reviewed evidence rows: **{len(reviewed_classifications):,}**
- Office-challenge claims: **{len(office_claim_validation):,}**
- Positive BASELINE/UFR/ROT evidence: **{format_money(office_challenge_evidence['five_year_total'].sum())}**
- Average AI-directness score: **{average_ai_score}**
"""))

In [ ]:
status_counts = office_claim_validation["validation_status"].value_counts().reindex(["validated", "needs_review", "unfunded"], fill_value=0)
status_funding = office_claim_validation.groupby("validation_status")["five_year_total"].sum().reindex(["validated", "needs_review", "unfunded"], fill_value=0.0)

executive_summary = pd.DataFrame(
    {
        "claim_count": status_counts,
        "five_year_total": status_funding.map(format_money),
        "five_year_total_display": status_funding.map(format_money),
    }
)
display(executive_summary)

ax = status_counts.plot(kind="bar", color=["#2f7d59", "#b1792f", "#9a4b45"], rot=0)
ax.set_title("Office Claim Validation Status")
ax.set_xlabel("")
ax.set_ylabel("Claims")
plt.show()

## Lead Office Accountability

These tables and visuals focus only on Genesis Mission Challenge claims where a program office is identified as the lead. The execution check compares each lead claim against actual BASELINE, UFR, and ROT programming rows where that same office is funding work classified to the claimed challenge.


In [ ]:
LEAD_STATUS_COLORS = {
    "validated": "#2f7d59",
    "needs_review": "#b1792f",
    "unfunded": "#9a4b45",
}
FUNDING_LEVELS = ["BASELINE", "UFR", "ROT"]
FUNDING_LEVEL_COLORS = {"BASELINE": "#4c78a8", "UFR": "#f58518", "ROT": "#7c3aed"}
FY_COLUMNS = ["fy2027", "fy2028", "fy2029", "fy2030", "fy2031", "fy2032"]

lead_claims = office_claim_validation.loc[
    office_claim_validation["claim_role"].astype(str).str.casefold().eq("lead")
].copy()
lead_claims = lead_claims.rename(columns={"claimed_office": "lead_program_office"})
lead_claims["execution_signal"] = lead_claims["validation_status"].map(
    {
        "validated": "Executing with clear evidence",
        "needs_review": "Executing, review evidence",
        "unfunded": "No BASELINE/UFR/ROT evidence found",
    }
).fillna("Unknown")
lead_claims["five_year_total_display"] = lead_claims["five_year_total"].map(format_money)
lead_claims_display = lead_claims[
    [
        "challenge_name",
        "lead_program_office",
        "execution_signal",
        "validation_status",
        "evidence_row_count",
        "evidence_wbs_count",
        "five_year_total_display",
        "high_confidence_count",
        "medium_confidence_count",
        "low_confidence_count",
        "review_flag_count",
        "evidence_wbs_names",
    ]
].sort_values(["validation_status", "lead_program_office", "challenge_name"])

display(Markdown("### Which offices claim lead responsibility?"))
display(lead_claims_display)

lead_claim_matrix = lead_claims.pivot_table(
    index="challenge_name",
    columns="lead_program_office",
    values="validation_status",
    aggfunc=lambda values: "; ".join(sorted(set(values))),
    fill_value="",
)
display(Markdown("### Lead office by Genesis Mission Challenge"))
display(lead_claim_matrix)

if not lead_claims.empty:
    challenges_order = lead_claims["challenge_name"].drop_duplicates().tolist()
    offices_order = sorted(lead_claims["lead_program_office"].drop_duplicates())
    x_positions = {office: index for index, office in enumerate(offices_order)}
    y_positions = {challenge: index for index, challenge in enumerate(challenges_order)}
    max_amount = max(float(lead_claims["five_year_total"].max()), 1.0)

    fig, ax = plt.subplots(figsize=(12, max(4.5, 0.46 * len(challenges_order))))
    for status, status_rows in lead_claims.groupby("validation_status"):
        amounts = status_rows["five_year_total"].astype(float).clip(lower=0.0)
        sizes = 120 + (amounts / max_amount) * 900
        ax.scatter(
            status_rows["lead_program_office"].map(x_positions),
            status_rows["challenge_name"].map(y_positions),
            s=sizes,
            color=LEAD_STATUS_COLORS.get(status, "#6b7280"),
            alpha=0.82,
            edgecolor="white",
            linewidth=1.2,
            label=status.replace("_", " ").title(),
        )
    for _, row in lead_claims.iterrows():
        ax.text(
            x_positions[row["lead_program_office"]] + 0.08,
            y_positions[row["challenge_name"]],
            format_money(row["five_year_total"]),
            va="center",
            fontsize=8,
            color="#263238",
        )
    ax.set_xticks(range(len(offices_order)))
    ax.set_xticklabels(offices_order)
    ax.set_yticks(range(len(challenges_order)))
    ax.set_yticklabels(challenges_order)
    ax.set_xlim(-0.5, max(len(offices_order) - 0.5, 0.5) + 0.9)
    ax.set_title("Lead Program Office Claims vs. Actual BASELINE/UFR/ROT Evidence")
    ax.set_xlabel("Claimed lead program office")
    ax.set_ylabel("Genesis Mission Challenge")
    ax.grid(axis="x", color="#e5e7eb", linewidth=0.8)
    ax.legend(title="Execution signal", loc="lower right")
    plt.tight_layout()
    plt.show()


In [ ]:
positive_funding_rows = reviewed_classifications.loc[
    reviewed_classifications["funding_level"].astype(str).str.upper().isin(FUNDING_LEVELS)
].copy()

lead_project_rows = positive_funding_rows.merge(
    lead_claims[["challenge_name", "lead_program_office", "validation_status", "execution_signal"]],
    left_on=["predicted_challenge", "normalized_funding_office"],
    right_on=["challenge_name", "lead_program_office"],
    how="inner",
)

for column in [*FY_COLUMNS, "five_year_total"]:
    if column in lead_project_rows.columns:
        lead_project_rows[column] = pd.to_numeric(lead_project_rows[column], errors="coerce").fillna(0.0)
        lead_project_rows[f"{column}_display"] = lead_project_rows[column].map(format_money)

if lead_project_rows.empty:
    lead_project_table = pd.DataFrame(
        columns=[
            "lead_program_office",
            "challenge_name",
            "funding_level",
            "five_year_total_display",
            "wbs_name",
            "wbs_code",
            "source_row_number",
            "confidence",
            "review_flag",
            "scope_description",
        ]
    )
    lead_project_funding = pd.DataFrame(
        columns=["lead_program_office", "challenge_name", "funding_level", "five_year_total"]
    )
else:
    lead_project_table = lead_project_rows[
        [
            "lead_program_office",
            "challenge_name",
            "funding_level",
            "five_year_total_display",
            "fynsp_cycle",
            "fy2027_display",
            "fy2028_display",
            "fy2029_display",
            "fy2030_display",
            "fy2031_display",
            "fy2032_display",
            "wbs_name",
            "wbs_code",
            "source_row_number",
            "confidence",
            "review_flag",
            "scope_description",
        ]
    ].sort_values(
        ["lead_program_office", "challenge_name", "funding_level", "five_year_total_display"],
        ascending=[True, True, True, False],
    )
    lead_project_funding = lead_project_rows.groupby(
        ["lead_program_office", "challenge_name", "funding_level"],
        as_index=False,
    ).agg(
        project_rows=("programming_record_id", "nunique"),
        wbs_count=("wbs_name", "nunique"),
        five_year_total=("five_year_total", "sum"),
    )

funding_amount_pivot = lead_project_funding.pivot_table(
    index=["lead_program_office", "challenge_name"],
    columns="funding_level",
    values="five_year_total",
    aggfunc="sum",
    fill_value=0.0,
).reset_index()
for funding_level in FUNDING_LEVELS:
    if funding_level not in funding_amount_pivot.columns:
        funding_amount_pivot[funding_level] = 0.0

funding_count_pivot = lead_project_funding.pivot_table(
    index=["lead_program_office", "challenge_name"],
    columns="funding_level",
    values="project_rows",
    aggfunc="sum",
    fill_value=0,
).reset_index()
funding_count_pivot = funding_count_pivot.rename(
    columns={level: f"{level.lower()}_project_rows" for level in FUNDING_LEVELS}
)
funding_count_columns = [f"{level.lower()}_project_rows" for level in FUNDING_LEVELS]
for column in funding_count_columns:
    if column not in funding_count_pivot.columns:
        funding_count_pivot[column] = 0

lead_execution_summary = lead_claims[
    [
        "challenge_name",
        "lead_program_office",
        "execution_signal",
        "validation_status",
        "evidence_row_count",
        "evidence_wbs_count",
        "review_flag_count",
    ]
].merge(
    funding_amount_pivot,
    on=["lead_program_office", "challenge_name"],
    how="left",
).merge(
    funding_count_pivot,
    on=["lead_program_office", "challenge_name"],
    how="left",
)
for column in [*FUNDING_LEVELS, *funding_count_columns]:
    lead_execution_summary[column] = pd.to_numeric(
        lead_execution_summary[column], errors="coerce"
    ).fillna(0)
lead_execution_summary["lead_programmed_total"] = lead_execution_summary[FUNDING_LEVELS].sum(axis=1)
for funding_level in FUNDING_LEVELS:
    lead_execution_summary[f"{funding_level.lower()}_display"] = lead_execution_summary[funding_level].map(format_money)
lead_execution_summary["lead_programmed_total_display"] = lead_execution_summary[
    "lead_programmed_total"
].map(format_money)
lead_execution_summary_display = lead_execution_summary[
    [
        "challenge_name",
        "lead_program_office",
        "execution_signal",
        "validation_status",
        *[f"{level.lower()}_display" for level in FUNDING_LEVELS],
        "lead_programmed_total_display",
        *funding_count_columns,
        "evidence_wbs_count",
        "review_flag_count",
    ]
].sort_values(["validation_status", "lead_program_office", "challenge_name"])

display(Markdown("### Lead office BASELINE/UFR/ROT funding by claimed mission"))
display(lead_execution_summary_display)

bar_source = lead_execution_summary.copy()
bar_source["lead_mission"] = (
    bar_source["lead_program_office"] + " | " + bar_source["challenge_name"]
)
bar_plot = bar_source.set_index("lead_mission")[FUNDING_LEVELS]
bar_plot = bar_plot.loc[bar_plot.sum(axis=1).sort_values().index]
if bar_plot.sum(axis=1).gt(0).any():
    ax = bar_plot.plot(
        kind="barh",
        stacked=True,
        figsize=(12, max(4.5, 0.4 * len(bar_plot))),
        color=[FUNDING_LEVEL_COLORS[level] for level in FUNDING_LEVELS],
    )
    ax.set_title("BASELINE, UFR, and ROT Funding Executed by Claimed Lead Office")
    ax.set_xlabel("FYNSP total")
    ax.set_ylabel("Lead office | Genesis Mission Challenge")
    ax.xaxis.set_major_formatter(lambda value, _: format_money(value))
    ax.legend(title="Funding level")
    plt.tight_layout()
    plt.show()
else:
    display(Markdown("No BASELINE, UFR, or ROT funding rows were found for lead office claims."))

display(Markdown("### Lead office project-level evidence"))
display(lead_project_table)


In [ ]:
claim_matrix = office_claim_validation.pivot_table(
    index="claimed_office",
    columns="challenge_name",
    values="validation_status",
    aggfunc=lambda values: "; ".join(sorted(set(values))),
    fill_value="",
)
display(claim_matrix)

In [ ]:
challenge_display = challenge_coverage.copy()
challenge_display["five_year_total_display"] = challenge_display["five_year_total"].map(format_money)
display(
    challenge_display[
        [
            "challenge_name",
            "claim_count",
            "validated",
            "needs_review",
            "unfunded",
            "evidence_row_count",
            "evidence_wbs_count",
            "five_year_total_display",
            "claimed_offices",
            "actual_funding_offices",
        ]
    ]
)

In [ ]:
unclaimed_evidence = office_challenge_evidence.loc[~office_challenge_evidence["is_claimed"].astype(bool)].copy()
unclaimed_evidence["five_year_total_display"] = unclaimed_evidence["five_year_total"].map(format_money)
unclaimed_evidence = unclaimed_evidence.sort_values("five_year_total", ascending=False)
display(
    unclaimed_evidence[
        [
            "challenge_name",
            "normalized_funding_office",
            "evidence_row_count",
            "evidence_wbs_count",
            "five_year_total_display",
            "evidence_wbs_names",
        ]
    ]
)


In [ ]:
office_summary = office_claim_validation.copy()
office_summary["five_year_total_display"] = office_summary["five_year_total"].map(format_money)
for office, office_rows in office_summary.groupby("claimed_office"):
    display(Markdown(f"## {office}"))
    display(
        office_rows[
            [
                "challenge_name",
                "claim_role",
                "validation_status",
                "evidence_row_count",
                "evidence_wbs_count",
                "five_year_total_display",
                "evidence_wbs_names",
            ]
        ].sort_values(["validation_status", "challenge_name"])
    )

In [ ]:
reviewed_display = reviewed_classifications.copy()
reviewed_display["five_year_total_display"] = reviewed_display["five_year_total"].map(format_money)
display(
    reviewed_display[
        [
            "programming_record_id",
            "source_row_number",
            "wbs_name",
            "source_sub_office",
            "normalized_funding_office",
            "funding_level",
            "original_predicted_challenge",
            "predicted_challenge",
            "confidence",
            "review_flag",
            "ai_directness_score",
            "ai_directness_label",
            "ai_directness_confidence",
            "ai_directness_rationale",
            "sme_review_decision",
            "sme_override_applied",
            "sme_notes",
            "sme_reviewer",
            "sme_review_date",
            "five_year_total_display",
            "scope_description",
        ]
    ].sort_values(["predicted_challenge", "normalized_funding_office", "wbs_name"])
)

In [ ]:
from programming_ontology._dashboard_report import write_classification_dashboard_html

html = write_classification_dashboard_html(
    paths,
    reviewed_classifications=reviewed_classifications,
    office_claim_validation=office_claim_validation,
    challenge_coverage=challenge_coverage,
    office_challenge_evidence=office_challenge_evidence,
)
display(Markdown(f"Exported executive dashboard HTML to `{paths.CLASSIFICATION_DASHBOARD_HTML_PATH}`."))
